## Generate Output

notebook is for creating and testing output generation for the NBPL file and also the internal daily email file

In [30]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, date
from openpyxl import load_workbook
from xgboost import XGBRegressor # likely move to modelling module
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import sys
import os
import pandas as pd 
from pathlib import Path
from datetime import datetime
from openpyxl import load_workbook

notebook_dir = Path(os.getcwd())
parent_dir = str(notebook_dir.parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from scripts.data_pull_functions import login_google_cloud
from scripts.gather_historic_data import gather_historic_data
from scripts.gather_data_to_forecast import gather_data_to_forecast
import scripts.model_training_functions
import scripts.create_output_functions

In [31]:
import importlib
importlib.reload(scripts.model_training_functions)
importlib.reload(scripts.create_output_functions)
from scripts.create_output_functions import create_nbpl_file, create_next_day_gas_burn_file, earliest_full_gas_day_with_forecast
from scripts.model_training_functions import fit_xgboost_clarissa_version, generate_feed_forward_forecast

In [3]:
client = login_google_cloud(project_name="bepc-prj-energy-prod")

In [ ]:
lead_columns = ['datetime', 'site', 'year', 'month', 'day', 'hour', 'hour_end', 'day_of_week', 'gas_day', 'hourly_gas_burn_MMBtu', 'daily_gas_burn_MMBtu']

model_df = gather_historic_data(start='2023-01-01', end='2026-09-02',  #make sure to adjust dates so it is dynamic. Set to today's date
                         client=client, lead_columns=lead_columns, 
                         yes_username=os.getenv('YES_USERNAME'), yes_password = os.getenv('YES_PASSWORD'), 
                         save_output=True
                        )


Pulling forecast: 2023-01-01 ---> 2026-09-02
Pulling actuals: 2023-01-01 ---> 2026-09-02
A file containing the merged data has been saved to ./data/processed-data/historic_data_df.csv
Pulling forecast: 2026-07-01 ---> 2026-09-11
A file containing the future data to has been saved to ./data/processed-data/data_to_forecast_df.csv


In [14]:
forward_df = gather_data_to_forecast(forecast_start='2026-07-01', forecast_end='2026-09-11', # make sure the dates for these are dynamic. Will cause error if not updated. Forecast end needs to be at least 8 days in the future to capture the full gas week
                            yes_username=os.getenv('YES_USERNAME'), yes_password=os.getenv('YES_PASSWORD'), 
                            save_output=True
                            )

Pulling forecast: 2026-07-01 ---> 2026-09-11
A file containing the future data to has been saved to ./data/processed-data/data_to_forecast_df.csv


In [15]:
features = ["availability_mw", "load_forecast", "net_load_forecast", "wind_forecast", "temperature_forecast", "wind_speed_forecast", "total_offline_forecast", 
           "offline_ng_forecast", "offline_coal_forecast", "hour", "day_of_week", "month", "gas_lag_1", "gas_lag_24", "gas_lag_168", "gas_roll_24", "gas_roll_168"]


# test_start_date is just one week ago from the today's date
fit_results = fit_xgboost_clarissa_version(model_df, sites=model_df['site'].unique(), test_start_date='2026-08-25', features=features, target = 'hourly_gas_burn_MMBtu')

site_models = fit_results['models']

In [28]:
predictions = generate_feed_forward_forecast(historic_df=model_df, forward_df=forward_df, models=site_models, features=features, save_output=True)

A file containing predictions has been saved to G:/Trading/Forecasts/Daily Gas Burn Forecast by Site/Forecasts/Hourly Gas Burn by Site Forecasts - 2026-09-02.csv


In [17]:
predictions.info()

<class 'pandas.DataFrame'>
RangeIndex: 1080 entries, 0 to 1079
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   site                   1080 non-null   str           
 1   datetime               1080 non-null   datetime64[us]
 2   gasday                 1080 non-null   datetime64[us]
 3   hourly_gas_burn_MMBtu  1080 non-null   float64       
 4   HE                     1080 non-null   str           
 5   date                   1080 non-null   object        
 6   gasday_of_week         1080 non-null   str           
 7   effective_day_of_week  1080 non-null   str           
dtypes: datetime64[us](2), float64(1), object(1), str(4)
memory usage: 81.3+ KB


In [ ]:
pd.to_datetime('2026-09-02') == predictions[predictions['gasday']=='2026-09-02']

In [32]:
create_nbpl_file(predictions=predictions, file_date='2026-09-03')

A file with the NBPL hourly forecasts is saved to G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Forecasts\Hourly Gas Burn Forecast for 09.02.2026.xlsm


## Daily Email Sent by Wes/Sam

The email has an attachment with 3 sheets
1. *Summary*
    - Contains a table summarized by site and additional pricing information. The table is composed of information from the following two sheets. 
2. *Hourly Gas Use*
    - Contains the hourly gas burn for each site. The first day of the summary table on 'Summary' uses the last nine hours of the gas day for next date the beginning of the gas specified on the summary table comes from the DA Gas sheet. Example - if the summary table has 9/1/2026 as the first row, the gas day goes from 9/1/2026 HE10 to 9/2/2026 HE 9. The data for hours 9/1/2026 HE10 through 9.1/2026 HE24 come from DA Gas. The data for 9/2/2026 HE1 through 9/2/2026 HE9 come from Hourly Gas Use sheet
3. *DA Gas*
    - Contains the hourly gas burn for the beginning
    - this sheet might not be needed with the new forecasting process

In [9]:
def earliest_full_gas_day_with_forecast(predictions: pd.DataFrame):
    daily_record_counts = predictions.groupby('gasday').count()['site'].reset_index()
    full_gas_days = daily_record_counts[daily_record_counts['site']==max(daily_record_counts['site'])]['gasday']
    full_gas_days = pd.to_datetime(full_gas_days)

    return full_gas_days

In [9]:
predictions.groupby('gasday').count()['site'].reset_index()

,gasday,site
0,2026-09-01,45
1,2026-09-02,120
2,2026-09-03,120
3,2026-09-04,120
4,2026-09-05,120
5,2026-09-06,120
6,2026-09-07,120
7,2026-09-08,120
8,2026-09-09,75


In [ ]:
predictions.head()

In [19]:
predictions.drop(columns=['datetime', 'date','HE', 'effective_day_of_week']).groupby(by=['gasday', 'gasday_of_week', 'site']).sum().reset_index().pivot(columns='site', index=['gasday', 'gasday_of_week'], values='hourly_gas_burn_MMBtu').round().reset_index()

site,gasday,gasday_of_week,CGS,DCS,GGS,LCS,PGS
0,2026-09-01,Tue,0.0,14971.0,-63.0,14968.0,39796.0
1,2026-09-02,Wed,0.0,38250.0,18017.0,50741.0,130713.0
2,2026-09-03,Thu,0.0,39436.0,17386.0,50832.0,131489.0
3,2026-09-04,Fri,0.0,38730.0,16428.0,45614.0,126341.0
4,2026-09-05,Sat,19472.0,40101.0,16303.0,47314.0,127770.0
5,2026-09-06,Sun,19623.0,41098.0,16680.0,46853.0,137724.0
6,2026-09-07,Mon,14524.0,38565.0,17499.0,45728.0,134155.0
7,2026-09-08,Tue,0.0,37366.0,17532.0,48503.0,133397.0
8,2026-09-09,Wed,0.0,35185.0,18967.0,48329.0,139352.0
9,2026-09-10,Thu,0.0,22533.0,12826.0,29692.0,91934.0


In [12]:
predictions.drop(columns=['gasday', 'datetime', 'gasday_of_week']).head(24)#.pivot(index=['site', 'date', 'effective_day_of_week'], columns = 'HE', values='hourly_gas_burn_MMBtu').reset_index().round(2)

,site,hourly_gas_burn_MMBtu,HE,date,effective_day_of_week
0,CGS,0.0,HE01,2026-09-01,Tue
1,CGS,0.0,HE02,2026-09-01,Tue
2,CGS,0.0,HE03,2026-09-01,Tue
3,CGS,0.0,HE04,2026-09-01,Tue
4,CGS,0.0,HE05,2026-09-01,Tue
5,CGS,0.0,HE06,2026-09-01,Tue
6,CGS,0.0,HE07,2026-09-01,Tue
7,CGS,0.0,HE08,2026-09-01,Tue
8,CGS,0.0,HE09,2026-09-01,Tue
9,CGS,0.0,HE10,2026-09-01,Tue


In [ ]:
formatted_predictions

In [33]:
create_next_day_gas_burn_file(predictions)

A file with the next day gas burns is saved to ..\output\next-day-gas-burn\next_day_gas_burn_09.02.2026.xlsx


In [ ]:
next_day_gas_burn_wb = load_workbook('../output/next-day-gas-burn/next_day_gas_burn_template.xlsx', data_only=True)

In [ ]:
summary_ws = next_day_gas_burn_wb['Summary']
hourly_gas_use_ws = next_day_gas_burn_wb['Hourly Gas Use']

In [ ]:
predictions.columns

In [ ]:
predictions

In [ ]:
predictions.drop(columns=['datetime', 'date','HE']).head(11)

In [ ]:
forecasted_days = earliest_full_gas_day_with_forecast(predictions)
forecasted_days = forecasted_days[forecasted_days >  pd.Timestamp.now()]
forecasted_days

In [ ]:
formatted_predictions = (
    predictions
    .drop(columns=['datetime', 'date','HE'])
    .groupby(by=['gasday', 'day_of_week', 'site'])
    .sum()
    .reset_index()
    .pivot(columns='site', index=['gasday', 'day_of_week'], values='hourly_gas_burn_MMBtu')
    .round(0)
    .reset_index()
    )

print(formatted_predictions.info())

formatted_predictions = formatted_predictions.loc[formatted_predictions['gasday'].isin(forecasted_days)]
formatted_predictions = formatted_predictions.reindex(columns=['gasday', 'day_of_week', 'DCS', 'GGS', 'CGS', 'PGS', 'LCS'])

In [ ]:
formatted_predictions = (
    predictions
    .drop(columns=['datetime', 'date','HE', 'day_of_week'])
    .groupby(by=['gasday', 'site'])
    .sum()
    .reset_index()
    .pivot(columns='site', index='gasday', values='hourly_gas_burn_MMBtu')
    .reset_index()
    .round(0)
 )
formatted_predictions

In [ ]:
formatted_predictions.insert(1, "Day", formatted_predictions['gasday'].dt.strftime('%a'))

In [ ]:
formatted_predictions = formatted_predictions.loc[formatted_predictions['gasday'].isin(forecasted_days)]

In [ ]:
formatted_predictions = formatted_predictions.reindex(columns=['gasday', 'Day', 'DCS', 'GGS', 'CGS', 'PGS', 'LCS'])

In [ ]:
formatted_predictions

In [ ]:
formatted_predictions.iloc[0]

In [ ]:
for df_row, excel_row in enumerate(range(3, 9)):
    for df_col, excel_col in enumerate(range(3, 10)):
        summary_ws.cell(row=excel_row, column=excel_col, value=formatted_predictions.iloc[df_row, df_col])
        #print(formatted_predictions.iloc[df_row, df_col])

In [ ]:
summary_ws['E3'].value

In [ ]:
#next_day_gas_burn_wb.save('../output/next-day-gas-burn/testing.xlsx')
#next_day_gas_burn_wb.close()

In [ ]:
## start of the Hourly Gas Use sheet
"""
predictions['HE'] = 'HE' + (predictions['datetime'] + pd.Timedelta(value=1, unit='h')).dt.strftime('%H')
predictions['HE'] = np.where(predictions['HE'] == 'HE00', 'HE24', predictions['HE'])
predictions['date'] = predictions['datetime'].dt.date
predictions['day_of_week'] = predictions['datetime'].dt.strftime('%a')
"""

In [ ]:
hourly_predictions_formatted = (
    predictions
    .drop(columns=['gasday', 'datetime'])
    .pivot(index=['site', 'date', 'day_of_week'], columns = 'HE', values='hourly_gas_burn_MMBtu')
    .reset_index()
    .round(2)
)

In [ ]:
hourly_predictions_formatted.head(24)

In [ ]:
#site_order = ['GGS', 'CGS', 'DCS', 'PGS', 'LCS']
site_start_row = {'GGS': 3, 'CGS': 13, 'DCS': 23, 'PGS': 33, 'LCS': 43}

In [ ]:
for site in site_start_row:
   
    site_predictions = hourly_predictions_formatted[hourly_predictions_formatted['site']==site]
    start_row = site_start_row[site]
    for df_row, excel_row in enumerate(range(start_row, start_row + site_predictions.shape[0])):
        
        for df_col, excel_col in enumerate(range(4, 30)):
            #print(f'site: {site}    excel_row: {excel_row}   excel_col: {excel_col}   df_row: {hourly_predictions_formatted.iloc[df_row, df_col+1]}')
            hourly_gas_use_ws.cell(row=excel_row, column=excel_col, value=site_predictions.iloc[df_row, df_col+1])
            if excel_col == 4: 
                hourly_gas_use_ws.cell(row=excel_row, column=excel_col).number_format = "mm/dd/yyyy"
            else:  
                hourly_gas_use_ws.cell(row=excel_row, column=excel_col).number_format = "#,##0.0"

In [ ]:
#widening columns
for col in hourly_gas_use_ws.columns:
    max_len = 0
    col_letter = col[0].column_letter  # Get letter like 'A', 'B', etc.
    for cell in col:
        if cell.value is not None:
            # Check length of the cell string
            max_len = max(max_len, len(str(cell.value)))
            
    # Add a little padding (e.g., +3) so it isn't too tight
    hourly_gas_use_ws.column_dimensions[col_letter].width = max(max_len + .1, 10)

In [ ]:
next_day_gas_burn_wb.save('../output/next-day-gas-burn/testing.xlsx')


In [ ]:
next_day_gas_burn_wb.close()

In [ ]:
excel_row

In [ ]:
excel_row

In [ ]:
hourly_predictions_formatted.loc['GGS'].shape